In [3]:
# === RESUME SETUP (run first) ===
import os

# After you've done "Add Data > Notebook Output Files" with your last saved version,
# set this to the mounted path. Leave as None for a totally fresh run.
PREV_RUN_DIR = "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap" # e.g. "/kaggle/input/rog-ap-6"

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # cuts CUDA OOM from fragmentation

if PREV_RUN_DIR and os.path.exists(PREV_RUN_DIR):
    print("Previous run found at:", PREV_RUN_DIR)
else:
    print("No previous run mounted — starting fresh, or PREV_RUN_DIR not set yet.")

Previous run found at: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap


In [4]:
import torch
_torch_pin = f"torch=={torch.__version__.split('+')[0]}"
print("Keeping installed PyTorch:", _torch_pin)

# accelerate==0.33.0 caps numpy<2.0, which drags Kaggle's numpy back from 2.x
# to 1.26.4 -- but Kaggle's pandas wheel is built against numpy 2.x's C ABI,
# so that downgrade breaks pandas at import time ("numpy.dtype size changed").
# accelerate>=0.34 dropped that cap, so bump it and pin numpy explicitly so
# pip can't silently downgrade it again.
!pip install -q \
    --only-binary=transformers,tokenizers,peft,sentencepiece,accelerate,datasets,numpy,torch \
    "transformers==4.44.2" "tokenizers==0.19.1" "peft==0.12.0" \
    "sentencepiece==0.2.0" "accelerate==0.34.2" "datasets==2.20.0" \
    "numpy>=2.0,<2.1" "graph-walker==1.0.6" "{_torch_pin}"

Keeping installed PyTorch: torch==2.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 74.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.

### Confirmation of the Packages

In [5]:
import transformers, tokenizers, networkx
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("networkx:", networkx.__version__)

transformers: 4.44.2
tokenizers: 0.19.1
networkx: 3.6.1


### Cloning official REPO


In [6]:
# (optional) from huggingface_hub import login; login(token="<your_hf_token>")
import os, sys

REPO_DIR = "/kaggle/working/reasoning-on-graphs"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/RManLuo/reasoning-on-graphs.git {REPO_DIR}

sys.path.append(os.path.join(REPO_DIR, "src"))

# RQ1
# Objective 1: Baseline Establishment

## RoG-WebQSP Baseline

### Config

In [7]:
N_QUESTIONS = 50          # Step 1 checklist: "50-100 questions, not the whole dataset"
DATASET_NAME = "rmanluo/RoG-webqsp"
SPLIT = "test"
MODEL_PATH = "rmanluo/RoG"   # official pre-trained RoG checkpoint (planning + reasoning, same model)
N_BEAM = 3                 # matches the paper's K=3 (Section 5.4) and scripts/planning.sh
OUTPUT_DIR = "/kaggle/working/step1_baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Load Dataset Subset

In [8]:
from datasets import load_dataset

full_test = load_dataset(DATASET_NAME, split=SPLIT)
subset = full_test.select(range(min(N_QUESTIONS, len(full_test))))
print(f"Loaded {len(subset)} / {len(full_test)} WebQSP test questions")
print("Fields:", subset.column_names)
subset[0]

README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

data/test-00000-of-00002-9ee8d68f7d951e1(…):   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/2826 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/246 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1628 [00:00<?, ? examples/s]

Loaded 50 / 1628 WebQSP test questions
Fields: ['id', 'question', 'answer', 'q_entity', 'a_entity', 'graph', 'choices']


{'id': 'WebQTest-0',
 'question': 'what does jamaican people speak',
 'answer': ['Jamaican English', 'Jamaican Creole English Language'],
 'q_entity': ['Jamaica'],
 'a_entity': ['Jamaican English', 'Jamaican Creole English Language'],
 'graph': [['Jamaica',
   'meteorology.cyclone_affected_area.cyclones',
   'Tropical Storm Keith'],
  ['Jamaica',
   'location.statistical_region.prevalence_of_undernourisment',
   'g.12tb6gh4f'],
  ['Latoya Greaves', 'olympics.olympic_athlete.country', 'm.0k8nh0b'],
  ['Jamaica',
   'location.statistical_region.electricity_consumption_per_capita',
   'm.0nf4wmg'],
  ['Hurricane Hilda',
   'meteorology.tropical_cyclone.affected_areas',
   'Yucatán Peninsula'],
  ['m.0wj6j0d',
   'sports.competitor_competition_relationship.tournament',
   '2013 World Championships in Athletics'],
  ['Jamaica',
   'location.statistical_region.market_cap_of_listed_companies_as_percent_of_gdp',
   'g.1hhc3gxpy'],
  ['Jamaica',
   'location.statistical_region.energy_use_per_ca

### Load the model

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False, clean_up_tokenization_spaces=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto", torch_dtype=torch.float16)
model.eval()
print("Model loaded.")

tokenizer_config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/78.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

Model loaded.


### Decode Patch

In [10]:
_original_decode = tokenizer.decode  # stash pre-patch decode for later verification (Part 2)

import types

def _sp_decode(self, token_ids, skip_special_tokens=True, **kwargs):
    tokens = self.convert_ids_to_tokens(token_ids, skip_special_tokens=skip_special_tokens)
    return self.sp_model.decode(tokens)

tokenizer.decode = types.MethodType(_sp_decode, tokenizer)
print("Patched tokenizer.decode to use sp_model.decode directly.")

Patched tokenizer.decode to use sp_model.decode directly.


### Diagnostic

In [11]:
# Diagnostic -- run with your CURRENT tokenizer, no reinstall needed
print("Tokenizer class:", type(tokenizer))
print("Has sp_model:", hasattr(tokenizer, "sp_model"))

test = "location.location.languages_spoken"
ids = tokenizer.encode(test, add_special_tokens=False)
tokens = tokenizer.convert_ids_to_tokens(ids)
print("tokens:", tokens)
print("tokenizer.decode():   ", repr(tokenizer.decode(ids, clean_up_tokenization_spaces=True)))

if hasattr(tokenizer, "sp_model"):
    print("sp_model.decode() direct:", repr(tokenizer.sp_model.decode(tokens)))

Tokenizer class: <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>
Has sp_model: True
tokens: ['▁location', '.', 'location', '.', 'l', 'anguages', '_', 'sp', 'oken']
tokenizer.decode():    ' location.location.languages_spoken'
sp_model.decode() direct: ' location.location.languages_spoken'


### Importing official planning functions

In [12]:
from utils.utils import InstructFormater
from qa_prediction.gen_rule_path import generate_seq, parse_prediction, INSTRUCTION

prompter = InstructFormater(os.path.join(REPO_DIR, "prompts", "llama2.txt"))
print("Planning instruction:", repr(INSTRUCTION))

Planning instruction: 'Please generate a valid relation path that can be helpful for answering the following question: '


### Run Planning (relation-path generation)

In [13]:
import time, json
from tqdm.auto import tqdm

planning_records = []
for sample in tqdm(subset, desc="Planning"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    t0 = time.time()
    raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
    planning_time = time.time() - t0
    rel_paths = parse_prediction(raw_output["paths"])   # top-N_BEAM predicted relation paths
    planning_records.append({
        "id": sample["id"],
        "question": sample["question"],
        "q_entity": sample["q_entity"],
        "a_entity": sample["a_entity"],
        "graph": sample["graph"],
        "predicted_paths": rel_paths,
        "planning_time_sec": planning_time,
    })

print(f"Generated relation-path plans for {len(planning_records)} questions.")
print("Example plan:", planning_records[0]["predicted_paths"])

Planning:   0%|          | 0/50 [00:00<?, ?it/s]

Generated relation-path plans for 50 questions.
Example plan: [['location.country.languages_spoken'], ['language.human_language.countries_spoken_in'], ['location.country.official_language']]


### Instrumented BFS Wrapper

In [14]:
# Adds READ-ONLY per-hop frontier counters. Does NOT change which nodes get
# expanded or pruned -- traversal order and the relation-match pruning rule
# are copied verbatim from src/utils/graph_utils.py.
from collections import deque
from utils.graph_utils import build_graph, bfs_with_rule   # the OFFICIAL, unmodified function

def bfs_with_rule_instrumented(graph, start_node, target_rule):
    result_paths = []
    hop_frontier = [0] * len(target_rule)   # entities admitted into the frontier at each hop
    queue = deque([(start_node, [])])
    while queue:
        current_node, current_path = queue.popleft()
        if len(current_path) == len(target_rule):
            result_paths.append(current_path)
        if len(current_path) < len(target_rule):
            if current_node not in graph:
                continue
            hop_idx = len(current_path)
            for neighbor in graph.neighbors(current_node):
                rel = graph[current_node][neighbor]["relation"]
                if rel != target_rule[hop_idx] or len(current_path) > len(target_rule):
                    continue
                queue.append((neighbor, current_path + [(current_node, rel, neighbor)]))
                hop_frontier[hop_idx] += 1
    return result_paths, hop_frontier

### Sanity check (Show 0 mismatches)

In [15]:
mismatches = 0
checked = 0
for rec in planning_records:
    graph = build_graph(rec["graph"])
    for entity in rec["q_entity"]:
        for rule in rec["predicted_paths"]:
            checked += 1
            official_result = bfs_with_rule(graph, entity, rule)
            instrumented_result, _ = bfs_with_rule_instrumented(graph, entity, rule)
            if official_result != instrumented_result:
                mismatches += 1

print(f"Checked {checked} (entity, relation-path) calls. Mismatches: {mismatches}")
assert mismatches == 0, "Instrumented BFS diverges from official RoG retrieval -- STOP, do not trust downstream numbers."

Checked 150 (entity, relation-path) calls. Mismatches: 0


### Running Retrieval

In [16]:
def path_endpoints(paths):
    return set(p[-1][-1] for p in paths if len(p) > 0)

retrieval_records = []
for rec in tqdm(planning_records, desc="Retrieval"):
    graph = build_graph(rec["graph"])
    gold_answers = set(rec["a_entity"])
    plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
    for rule in plans:
        t0 = time.time()
        all_paths = []
        hop_frontier_total = [0] * len(rule)
        if len(rule) > 0:
            for entity in rec["q_entity"]:
                paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                all_paths.extend(paths)
                for h in range(len(rule)):
                    hop_frontier_total[h] += hop_frontier[h]
        retrieval_time = time.time() - t0
        reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
        retrieval_records.append({
            "question_id": rec["id"],
            "question": rec["question"],
            "topic_entity": rec["q_entity"],
            "predicted_relation_path": rule,
            "hop_frontiers": hop_frontier_total,   # [hop_1_frontier, hop_2_frontier, ...]
            "retrieved_paths_count": len(all_paths),
            "gold_answers": sorted(gold_answers),
            "reachable_gold_answers": reachable_gold,
            "gold_reachable": len(reachable_gold) > 0,
            "retrieval_time_sec": retrieval_time,
            "planning_time_sec": rec["planning_time_sec"],
        })

print(f"{len(retrieval_records)} (question, relation-plan) retrieval records collected.")

Retrieval:   0%|          | 0/50 [00:00<?, ?it/s]

147 (question, relation-plan) retrieval records collected.


### Question Evaluation

In [17]:
import pandas as pd
df = pd.DataFrame(retrieval_records)

question_coverage = (
    df.groupby("question_id")["gold_reachable"]
      .any()
)

print("Questions evaluated:", len(question_coverage))
print("Questions reaching at least one gold answer:", question_coverage.sum())
print("Question-level retrieval coverage:",
      question_coverage.mean() * 100)

Questions evaluated: 50
Questions reaching at least one gold answer: 38
Question-level retrieval coverage: 76.0


### Computing missed_qids

In [18]:
from collections import defaultdict

reachable_by_question = defaultdict(bool)
for rec in retrieval_records:
    reachable_by_question[rec["question_id"]] |= rec["gold_reachable"]

all_qids = {rec["question_id"] for rec in retrieval_records}
missed_qids = [qid for qid in all_qids if not reachable_by_question[qid]]

print(f"{len(missed_qids)} / {len(all_qids)} questions missed")

12 / 50 questions missed


### Revised Failure Analysis

In [19]:
from collections import deque

def find_gold_paths_official_graph(rec, max_hops=2):
    G = build_graph(rec["graph"])
    gold = set(rec["a_entity"])
    found = []

    for start in rec["q_entity"]:
        queue = deque([(start, [], [start])])

        while queue:
            node, relations, entities = queue.popleft()

            if len(relations) > 0 and node in gold:
                found.append({
                    "relations": relations,
                    "entities": entities
                })

            if len(relations) == max_hops:
                continue

            if node not in G:
                continue

            for neighbor in G.neighbors(node):
                rel = G[node][neighbor]["relation"]

                queue.append((
                    neighbor,
                    relations + [rel],
                    entities + [neighbor]
                ))

    return found

for qid in missed_qids:

    rec = next(r for r in planning_records if r["id"] == qid)

    gold_paths = find_gold_paths_official_graph(rec, max_hops=2)

    predicted = {
        tuple(p) for p in rec["predicted_paths"]
    }

    gold_relation_paths = {
        tuple(p["relations"]) for p in gold_paths
    }

    print("\n" + "="*100)
    print("QUESTION:", rec["question"])
    print("GOLD:", rec["a_entity"])

    print("\nPredicted:")
    for p in predicted:
        print(" ", " -> ".join(p))

    print("\nGold-supporting paths in OFFICIAL BUILT GRAPH:")
    for p in gold_paths[:10]:
        print(
            " ",
            " -> ".join(p["entities"]),
            "\n    ",
            " -> ".join(p["relations"])
        )

    if not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"

    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"

    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    print("\nDIAGNOSIS:", diagnosis)


QUESTION: who plays ken barlow in coronation street
GOLD: ['William Roache']

Predicted:
  tv.tv_program.country_of_origin -> people.person.nationality
  tv.regular_tv_appearance.series -> tv.regular_tv_appearance.actor
  tv.regular_tv_appearance.series -> tv.tv_actor.starring_roles

Gold-supporting paths in OFFICIAL BUILT GRAPH:

DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: who is governor of ohio 2011
GOLD: ['John Kasich', 'Ted Strickland', 'Return J. Meigs, Jr.']

Predicted:
  government.governmental_jurisdiction.governing_officials -> government.government_position_held.office_holder
  government.government_position_held.jurisdiction_of_office -> government.government_position_held.office_holder
  government.governmental_jurisdiction.governing_officials -> government.politician.government_positions_held

Gold-supporting paths in OFFICIAL BUILT GRAPH:
  Ohio -> United States of America -> Return J. Meigs, Jr. 
     base.locations.states_and_provences.country -> people.pers

### JSON, CSV save

In [20]:
import pandas as pd

json_path = os.path.join(OUTPUT_DIR, "step1_baseline_webqsp.json")
csv_path = os.path.join(OUTPUT_DIR, "step1_baseline_webqsp.csv")

with open(json_path, "w") as f:
    json.dump(retrieval_records, f, indent=2)

df = pd.DataFrame(retrieval_records)
df.to_csv(csv_path, index=False)

print("Saved:", json_path)
print("Saved:", csv_path)
df[["retrieved_paths_count", "gold_reachable", "retrieval_time_sec"]].describe()

Saved: /kaggle/working/step1_baseline/step1_baseline_webqsp.json
Saved: /kaggle/working/step1_baseline/step1_baseline_webqsp.csv


,retrieved_paths_count,retrieval_time_sec
count,147.000000,147.000000
mean,6.564626,0.000197
std,27.793531,0.000231
min,0.000000,0.000007
25%,0.000000,0.000056
50%,1.000000,0.000141
75%,3.000000,0.000217
max,320.000000,0.001233


### Worked Example

In [21]:
example = next(r for r in retrieval_records if len(r["predicted_relation_path"]) > 0)

print("Question:")
print(example["question"])
print()
print("Topic entity:")
print(example["topic_entity"])
print()
print("RoG relation plan:")
print(" -> ".join(example["predicted_relation_path"]))
print()
for i, count in enumerate(example["hop_frontiers"], start=1):
    print(f"Hop {i}:")
    print(f"{count} candidate entities")
    print()
print("Retrieved paths:")
print(example["retrieved_paths_count"])
print()
print("Gold answer reachable:")
print("Yes" if example["gold_reachable"] else "No")
print()
print("Retrieval time:")
print(f"{example['retrieval_time_sec']*1000:.1f} ms")

Question:
what does jamaican people speak

Topic entity:
['Jamaica']

RoG relation plan:
location.country.languages_spoken

Hop 1:
1 candidate entities

Retrieved paths:
1

Gold answer reachable:
Yes

Retrieval time:
1.2 ms


### Final RoG Answer

In [22]:
from qa_prediction.build_qa_input import PromptBuilder

reasoning_prompter = PromptBuilder(
    os.path.join(REPO_DIR, "prompts", "llama2_predict.txt"),
    add_rule=True,
    maximun_token=4096 - 100,
    tokenize=lambda t: len(tokenizer.tokenize(t)),
)

by_question = {}
for rec in planning_records:
    by_question[rec["id"]] = rec

final_answers = {}
for qid, rec in tqdm(by_question.items(), desc="Reasoning"):
    q_dict = {
        "question": rec["question"],
        "graph": rec["graph"],
        "q_entity": rec["q_entity"],
        "predicted_paths": rec["predicted_paths"],
        "choices": [],
    }
    prompt = reasoning_prompter.process_input(q_dict)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
    gen_time = time.time() - t0
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    final_answers[qid] = {"final_RoG_answer": text, "reasoning_time_sec": gen_time}

for rec in retrieval_records:
    rec.update(final_answers.get(rec["question_id"], {}))

with open(json_path, "w") as f:
    json.dump(retrieval_records, f, indent=2)
pd.DataFrame(retrieval_records).to_csv(csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

Reasoning:   0%|          | 0/50 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Verify the Decode Patch(full set)

In [23]:
default_decode_records = []
for sample in tqdm(subset, desc="Verify (default decode)"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids, num_beams=N_BEAM, num_return_sequences=N_BEAM,
            early_stopping=False, do_sample=True, return_dict_in_generate=True,
            output_scores=True, max_new_tokens=100,
        )
    raw_sequences = output.sequences[:, input_ids.shape[1]:]
    default_text = [_original_decode(seq, skip_special_tokens=True).strip() for seq in raw_sequences]
    default_paths = parse_prediction(default_text)
    default_decode_records.append({"id": sample["id"], "predicted_paths": default_paths})

by_id_default = {r["id"]: r["predicted_paths"] for r in default_decode_records}
diffs = 0
examples_shown = 0
for rec in planning_records:
    patched = rec["predicted_paths"]
    default = by_id_default.get(rec["id"], [])
    if patched != default:
        diffs += 1
        if examples_shown < 5:
            print(f"\nQID {rec['id']}  --  {rec['question']}")
            print("  patched (sp_model.decode): ", patched)
            print("  default (tokenizer.decode):", default)
            examples_shown += 1

print(f"\n{diffs} / {len(planning_records)} questions differ between patched and default decode.")
if diffs == 0:
    print("No difference on the pilot -- the patch is a safe no-op here; keep it for robustness.")
else:
    print("Inspect the examples above: genuine fix, or a new deviation? Decide before scaling.")

Verify (default decode):   0%|          | 0/50 [00:00<?, ?it/s]


0 / 50 questions differ between patched and default decode.
No difference on the pilot -- the patch is a safe no-op here; keep it for robustness.


### Runtime Estimate

In [24]:
df_pilot = pd.DataFrame(retrieval_records)
mean_plan = df_pilot["planning_time_sec"].mean()
mean_retr = df_pilot["retrieval_time_sec"].mean()
n_full = 1628  # official WebQSP test size (RoG paper, Table 6)

est_plan_sec = mean_plan * n_full
est_retr_sec = mean_retr * n_full
est_total_hr = (est_plan_sec + est_retr_sec) / 3600

print(f"Mean planning time/question:  {mean_plan:.2f} s")
print(f"Mean retrieval time/question: {mean_retr:.3f} s")
print(f"Projected for {n_full} questions:")
print(f"  Planning:  ~{est_plan_sec/60:.1f} min")
print(f"  Retrieval: ~{est_retr_sec/60:.1f} min")
print(f"  Total:     ~{est_total_hr:.2f} GPU-hours (planning+retrieval only, not the optional final-answer pass)")
print()
print("Kaggle sessions have historically capped continuous runtime around 9 hours,")
print("with a weekly GPU quota that's fluctuated between ~30 and ~40 hours. Check")
print("your account's current quota (Settings) -- it changes over time. The")
print("checkpointed loops below handle a session cutoff gracefully: just re-run")
print("the cell and it resumes from the last completed question.")

Mean planning time/question:  1.99 s
Mean retrieval time/question: 0.000 s
Projected for 1628 questions:
  Planning:  ~54.0 min
  Retrieval: ~0.0 min
  Total:     ~0.90 GPU-hours (planning+retrieval only, not the optional final-answer pass)

Kaggle sessions have historically capped continuous runtime around 9 hours,
with a weekly GPU quota that's fluctuated between ~30 and ~40 hours. Check
your account's current quota (Settings) -- it changes over time. The
checkpointed loops below handle a session cutoff gracefully: just re-run
the cell and it resumes from the last completed question.


### Full-scale Config

In [25]:
FULL_OUTPUT_DIR = "/kaggle/working/step1_baseline_full"
os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)
# === RESTORE WebQSP checkpoints from last session ===
import shutil

if PREV_RUN_DIR:
    prev_dir = os.path.join(PREV_RUN_DIR, "step1_baseline_full")
    if os.path.exists(prev_dir):
        for fname in os.listdir(prev_dir):
            dst = os.path.join(FULL_OUTPUT_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(prev_dir, fname), dst)
                print("Restored:", fname)
    else:
        print("No prior WebQSP full-run folder in PREV_RUN_DIR.")
else:
    print("PREV_RUN_DIR not set — nothing to restore.")

PLANNING_CKPT = os.path.join(FULL_OUTPUT_DIR, "planning_webqsp_full.jsonl")
REASON_CKPT = os.path.join(FULL_OUTPUT_DIR, "reasoning_webqsp_full.jsonl")

full_test_all = load_dataset(DATASET_NAME, split=SPLIT)  # all 1,628 WebQSP test questions
print(f"Full test set: {len(full_test_all)} questions")

Full test set: 1628 questions


### Checkpointed planning (full set)

In [26]:
def load_checkpoint(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    done[rec["id"]] = rec
    return done

planning_done = load_checkpoint(PLANNING_CKPT)
print(f"Resuming: {len(planning_done)} questions already planned.")

with open(PLANNING_CKPT, "a") as fout:
    for sample in tqdm(full_test_all, desc="Planning (full)"):
        if sample["id"] in planning_done:
            continue
        input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
        t0 = time.time()
        raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=False, max_new_tokens=100)
        planning_time = time.time() - t0
        rel_paths = parse_prediction(raw_output["paths"])
        rec = {
            "id": sample["id"], "question": sample["question"],
            "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
            "graph": sample["graph"], "predicted_paths": rel_paths,
            "planning_time_sec": planning_time,
        }
        fout.write(json.dumps(rec) + "\n")
        fout.flush()
        planning_done[sample["id"]] = rec

planning_records_full = list(planning_done.values())
print(f"Planning complete: {len(planning_records_full)} / {len(full_test_all)} questions.")

Resuming: 1628 questions already planned.


Planning (full):   0%|          | 0/1628 [00:00<?, ?it/s]

Planning complete: 1628 / 1628 questions.


### Retrieval (Full set)

In [27]:
retrieval_records_full = []
for rec in tqdm(planning_records_full, desc="Retrieval (full)"):
    graph = build_graph(rec["graph"])
    gold_answers = set(rec["a_entity"])
    plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
    for rule in plans:
        t0 = time.time()
        all_paths = []
        hop_frontier_total = [0] * len(rule)
        if len(rule) > 0:
            for entity in rec["q_entity"]:
                paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                all_paths.extend(paths)
                for h in range(len(rule)):
                    hop_frontier_total[h] += hop_frontier[h]
        retrieval_time = time.time() - t0
        reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
        retrieval_records_full.append({
            "question_id": rec["id"], "question": rec["question"],
            "topic_entity": rec["q_entity"], "predicted_relation_path": rule,
            "hop_frontiers": hop_frontier_total,
            "retrieved_paths_count": len(all_paths),
            "gold_answers": sorted(gold_answers),
            "reachable_gold_answers": reachable_gold,
            "gold_reachable": len(reachable_gold) > 0,
            "retrieval_time_sec": retrieval_time,
            "planning_time_sec": rec["planning_time_sec"],
        })

print(f"{len(retrieval_records_full)} retrieval records (full set) collected.")

Retrieval (full):   0%|          | 0/1628 [00:00<?, ?it/s]

4838 retrieval records (full set) collected.


### Question-Level Coverage

In [28]:
df_full = pd.DataFrame(retrieval_records_full)
question_coverage_full = df_full.groupby("question_id")["gold_reachable"].any()
print("Questions evaluated:", len(question_coverage_full))
print("Questions reaching >=1 gold answer:", question_coverage_full.sum())
print("Question-level retrieval coverage: %.2f%%" % (question_coverage_full.mean() * 100))

Questions evaluated: 1628
Questions reaching >=1 gold answer: 1369
Question-level retrieval coverage: 84.09%


### Aggregated Failure Analysis

In [29]:
from collections import Counter

reachable_by_q_full = defaultdict(bool)
for rec in retrieval_records_full:
    reachable_by_q_full[rec["question_id"]] |= rec["gold_reachable"]

all_qids_full = {rec["question_id"] for rec in retrieval_records_full}
missed_qids_full = [q for q in all_qids_full if not reachable_by_q_full[q]]
print(f"{len(missed_qids_full)} / {len(all_qids_full)} questions missed")

planning_by_id_full = {r["id"]: r for r in planning_records_full}
diagnosis_counts = Counter()
diagnosis_by_qid = {}
SAMPLE_PRINT = 5
printed = 0

for qid in tqdm(missed_qids_full, desc="Failure analysis"):
    rec = planning_by_id_full[qid]
    gold_paths = find_gold_paths_official_graph(rec, max_hops=2)  # WebQSP max hop = 2 (Table 6); bump to 4 for CWQ later
    predicted = {tuple(p) for p in rec["predicted_paths"]}
    gold_relation_paths = {tuple(p["relations"]) for p in gold_paths}

    if not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"
    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"
    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    diagnosis_counts[diagnosis] += 1
    diagnosis_by_qid[qid] = diagnosis

    if printed < SAMPLE_PRINT:
        print("\n" + "="*100)
        print("QUESTION:", rec["question"])
        print("DIAGNOSIS:", diagnosis)
        printed += 1

print("\n--- Failure breakdown ---")
for k, v in diagnosis_counts.most_common():
    print(f"{k}: {v}  ({100*v/max(len(missed_qids_full),1):.1f}% of missed questions)")

for rec in retrieval_records_full:
    rec["failure_diagnosis"] = diagnosis_by_qid.get(rec["question_id"])

259 / 1628 questions missed


Failure analysis:   0%|          | 0/259 [00:00<?, ?it/s]


QUESTION: when was the printing press invented by gutenberg
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: what sarah dessen books are movies
DIAGNOSIS: PLANNER FAILURE

QUESTION: when was lucy lawless born
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: who was selena gomez in barney and friends
DIAGNOSIS: PLANNER FAILURE

QUESTION: when did the burma cyclone happen
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

--- Failure breakdown ---
PLANNER FAILURE: 188  (72.6% of missed questions)
OFFICIAL GRAPH COVERAGE FAILURE: 71  (27.4% of missed questions)


### Full Scale Output

In [30]:
full_json_path = os.path.join(FULL_OUTPUT_DIR, "step1_baseline_webqsp_full.json")
full_csv_path = os.path.join(FULL_OUTPUT_DIR, "step1_baseline_webqsp_full.csv")

with open(full_json_path, "w") as f:
    json.dump(retrieval_records_full, f, indent=2)
pd.DataFrame(retrieval_records_full).to_csv(full_csv_path, index=False)
print("Saved:", full_json_path)
print("Saved:", full_csv_path)

Saved: /kaggle/working/step1_baseline_full/step1_baseline_webqsp_full.json
Saved: /kaggle/working/step1_baseline_full/step1_baseline_webqsp_full.csv


### Final RoG Answer (Full Planning)

In [31]:
reasoning_done = load_checkpoint(REASON_CKPT)
print(f"Resuming: {len(reasoning_done)} answers already generated.")

with open(REASON_CKPT, "a") as fout:
    for qid, rec in tqdm(planning_by_id_full.items(), desc="Reasoning (full)"):
        if qid in reasoning_done:
            continue
        q_dict = {
            "question": rec["question"], "graph": rec["graph"],
            "q_entity": rec["q_entity"], "predicted_paths": rec["predicted_paths"],
            "choices": [],
        }
        prompt = reasoning_prompter.process_input(q_dict)
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        t0 = time.time()
        with torch.inference_mode():
            out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
        gen_time = time.time() - t0
        text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        out_rec = {"id": qid, "final_RoG_answer": text, "reasoning_time_sec": gen_time}
        fout.write(json.dumps(out_rec) + "\n")
        fout.flush()
        reasoning_done[qid] = out_rec

for rec in retrieval_records_full:
    ans = reasoning_done.get(rec["question_id"])
    if ans:
        rec.update({"final_RoG_answer": ans["final_RoG_answer"], "reasoning_time_sec": ans["reasoning_time_sec"]})

with open(full_json_path, "w") as f:
    json.dump(retrieval_records_full, f, indent=2)
pd.DataFrame(retrieval_records_full).to_csv(full_csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

Resuming: 1628 answers already generated.


Reasoning (full):   0%|          | 0/1628 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Confirm the full reasoning loop 

In [32]:
missing = [qid for qid in planning_by_id_full if qid not in reasoning_done]
print(f"{len(reasoning_done)} / {len(planning_by_id_full)} questions have a final answer.")
if missing:
    print(f"{len(missing)} still missing -- re-run the Cell 54 reasoning loop (it resumes automatically).")
else:
    print("Reasoning complete. Safe to proceed.")

1628 / 1628 questions have a final answer.
Reasoning complete. Safe to proceed.


### Memory Cache saved for reasoning

In [33]:
# === MEMORY: drop cached subgraphs now that WebQSP reasoning is done ===
import gc
for rec in planning_records_full:
    rec.pop("graph", None)
gc.collect()
print("Dropped cached subgraphs from planning_records_full to free RAM.")

Dropped cached subgraphs from planning_records_full to free RAM.


### Clean per-question table

In [34]:
from collections import defaultdict
import pandas as pd

retrieval_time_by_qid = defaultdict(float)
gold_reachable_by_qid = defaultdict(bool)
for rec in retrieval_records_full:
    retrieval_time_by_qid[rec["question_id"]] += rec["retrieval_time_sec"]
    gold_reachable_by_qid[rec["question_id"]] |= rec["gold_reachable"]

rows = []
for qid, prec in planning_by_id_full.items():
    rrow = reasoning_done.get(qid, {})
    rows.append({
        "id": qid,
        "question": prec["question"],
        "gold_answers": prec["a_entity"],
        "predicted_paths": prec["predicted_paths"],
        "planning_time_sec": prec["planning_time_sec"],
        "retrieval_time_sec": retrieval_time_by_qid.get(qid, 0.0),
        "reasoning_time_sec": rrow.get("reasoning_time_sec"),
        "final_RoG_answer": rrow.get("final_RoG_answer"),
        "gold_reachable": gold_reachable_by_qid.get(qid, False),
    })

per_question = pd.DataFrame(rows)
n = len(per_question)
assert per_question["reasoning_time_sec"].notna().all(), "Some questions missing reasoning -- finish Cell 54 first."
print(f"Per-question baseline table: {n} questions.")

Per-question baseline table: 1628 questions.


### Full per-question table

In [35]:
pd.set_option("display.max_colwidth", 100)  

display_cols = [
    "id", "question", "gold_answers", "predicted_paths",
    "planning_time_sec", "retrieval_time_sec", "reasoning_time_sec",
    "final_RoG_answer", "gold_reachable"
]

per_question_display = per_question[display_cols].rename(columns={"id": "question_id"})
per_question_display

,question_id,question,gold_answers,predicted_paths,planning_time_sec,retrieval_time_sec,reasoning_time_sec,final_RoG_answer,gold_reachable
0,WebQTest-0,what does jamaican people speak,"[Jamaican English, Jamaican Creole English Language]","[[location.country.languages_spoken], [language.human_language.countries_spoken_in], [location.c...",1.324136,0.003920,0.893439,Jamaican English\nJamaican Creole English Language,True
1,WebQTest-1,what did james k polk do before he was president,"[United States Representative, Governor of Tennessee, Speaker of the United States House of Repr...","[[government.government_position_held.office_holder, government.government_position_held.office_...",2.938305,0.000676,0.528926,United States Representative,True
2,WebQTest-3,who plays ken barlow in coronation street,[William Roache],"[[tv.tv_program.country_of_origin, people.person.nationality], [tv.regular_tv_appearance.series,...",1.790577,0.000319,0.472994,David Hanson,False
3,WebQTest-6,where is jamarcus russell from,[Mobile],"[[location.location.people_born_here], [people.person.place_of_birth], [people.person.nationality]]",1.032935,0.000264,0.461035,United States of America\nMobile,True
4,WebQTest-7,where was george washington carver from,[Diamond],"[[people.person.place_of_birth], [location.location.people_born_here], [people.person.nationality]]",1.034217,0.000151,0.583327,United States of America\nDiamond,True
...,...,...,...,...,...,...,...,...,...
1623,WebQTest-2027,what team did david beckham play for before la galaxy,[Manchester United F.C.],"[[sports.pro_athlete.teams, sports.sports_team_roster.team], [soccer.football_player.statistics,...",1.927255,0.000418,1.879107,Paris Saint-Germain F.C.\nA.C. Milan\nManchester United F.C.,True
1624,WebQTest-2028,who is the current leader of france 2010,[Nicolas Sarkozy],"[[people.person.nationality], [base.onephylogeny.type_of_thing.things_of_this_type, people.perso...",2.396295,0.003479,0.968816,Nicolas Sarkozy\nFrançois Hollande,True
1625,WebQTest-2029,where was the palace of knossos located,"[Crete, Greece]","[[location.location.containedby], [architecture.building.building_complex], [architecture.buildi...",0.821460,0.000087,0.154127,Greece,True
1626,WebQTest-2030,where is roswell area 51,"[Lincoln County, Nevada]","[[aviation.airport.serves], [location.location.containedby], [location.location.contains]]",0.892296,0.000057,0.214914,Lincoln County,True


### Saved as CSV and JSON

In [36]:
csv_out = os.path.join(FULL_OUTPUT_DIR, "webqsp_per_question_baseline.csv")
json_out = os.path.join(FULL_OUTPUT_DIR, "webqsp_per_question_baseline.json")

per_question_display.to_csv(csv_out, index=False)
per_question_display.to_json(json_out, orient="records", indent=2)

print("Saved:", csv_out)
print("Saved:", json_out)

Saved: /kaggle/working/step1_baseline_full/webqsp_per_question_baseline.csv
Saved: /kaggle/working/step1_baseline_full/webqsp_per_question_baseline.json


### Timing Statistics

In [37]:
def total_avg(col):
    total = per_question[col].sum()
    return total, total / n

plan_total, plan_avg = total_avg("planning_time_sec")
retr_total, retr_avg = total_avg("retrieval_time_sec")
reas_total, reas_avg = total_avg("reasoning_time_sec")
e2e_total = plan_total + retr_total + reas_total
e2e_avg = e2e_total / n

print(f"Total planning time:   {plan_total:9.1f} s  ({plan_total/60:7.2f} min)   avg/question: {plan_avg:.3f} s")
print(f"Total retrieval time:  {retr_total:9.1f} s  ({retr_total/60:7.2f} min)   avg/question: {retr_avg:.4f} s")
print(f"Total reasoning time:  {reas_total:9.1f} s  ({reas_total/60:7.2f} min)   avg/question: {reas_avg:.3f} s")
print(f"Total end-to-end time: {e2e_total:9.1f} s  ({e2e_total/60:7.2f} min)   avg/question: {e2e_avg:.3f} s")

Total planning time:      2634.3 s  (  43.91 min)   avg/question: 1.618 s
Total retrieval time:        1.2 s  (   0.02 min)   avg/question: 0.0008 s
Total reasoning time:     5148.2 s  (  85.80 min)   avg/question: 3.162 s
Total end-to-end time:    7783.7 s  ( 129.73 min)   avg/question: 4.781 s


### Retrieval Coverage and official Hit/Hits@1/F1

In [38]:
import sys
sys.path.append(os.path.join(REPO_DIR, "src"))
from qa_prediction.evaluate_results import eval_acc, eval_hit, eval_f1

n_covered = int(per_question["gold_reachable"].sum())
coverage_pct = 100 * n_covered / n

hit_list, acc_list, f1_list, prec_list, rec_list = [], [], [], [], []
for _, row in per_question.iterrows():
    prediction = [p for p in row["final_RoG_answer"].split("\n") if p.strip()]
    prediction_str = " ".join(prediction)
    f1, precision, recall = eval_f1(prediction, row["gold_answers"])
    hit_list.append(eval_hit(prediction_str, row["gold_answers"]))
    acc_list.append(eval_acc(prediction_str, row["gold_answers"]))
    f1_list.append(f1); prec_list.append(precision); rec_list.append(recall)

hits1_pct = 100 * sum(hit_list) / n
f1_pct = 100 * sum(f1_list) / n

print(f"Questions reaching >=1 gold: {n_covered}")
print(f"Question-level retrieval coverage: {coverage_pct:.2f}%")
print(f"Hit/Hits@1: {hits1_pct:.2f}%")
print(f"F1: {f1_pct:.2f}%")

Questions reaching >=1 gold: 1369
Question-level retrieval coverage: 84.09%
Hit/Hits@1: 85.93%
F1: 70.27%


### Freezing Baseline

In [39]:
baseline_summary = f"""Dataset: WebQSP
Test questions: {n}

Retrieval:
  Questions reaching >=1 gold: {n_covered}
  Question-level retrieval coverage: {coverage_pct:.2f}%

Timing:
  Total planning time:   {plan_total:.1f} s ({plan_total/60:.2f} min)   Avg/question: {plan_avg:.3f} s
  Total retrieval time:  {retr_total:.1f} s ({retr_total/60:.2f} min)   Avg/question: {retr_avg:.4f} s
  Total reasoning time:  {reas_total:.1f} s ({reas_total/60:.2f} min)   Avg/question: {reas_avg:.3f} s
  Total end-to-end time: {e2e_total:.1f} s ({e2e_total/60:.2f} min)   Avg/question: {e2e_avg:.3f} s

Final QA:
  Hit/Hits@1: {hits1_pct:.2f}%
  F1: {f1_pct:.2f}%
"""
print(baseline_summary)

with open(os.path.join(FULL_OUTPUT_DIR, "webqsp_baseline_summary.txt"), "w") as f:
    f.write(baseline_summary)

per_question.to_json(os.path.join(FULL_OUTPUT_DIR, "webqsp_baseline_predictions.json"), orient="records", indent=2)
per_question.to_csv(os.path.join(FULL_OUTPUT_DIR, "webqsp_baseline_predictions.csv"), index=False)
print("Baseline frozen.")

Dataset: WebQSP
Test questions: 1628

Retrieval:
  Questions reaching >=1 gold: 1369
  Question-level retrieval coverage: 84.09%

Timing:
  Total planning time:   2634.3 s (43.91 min)   Avg/question: 1.618 s
  Total retrieval time:  1.2 s (0.02 min)   Avg/question: 0.0008 s
  Total reasoning time:  5148.2 s (85.80 min)   Avg/question: 3.162 s
  Total end-to-end time: 7783.7 s (129.73 min)   Avg/question: 4.781 s

Final QA:
  Hit/Hits@1: 85.93%
  F1: 70.27%

Baseline frozen.


### Memory clear of RoG-WebQSP

In [40]:
# === MEMORY: clear WebQSP intermediates before starting CWQ ===
import torch
for _name in ["retrieval_records_full", "retrieval_records", "planning_records",
              "df_full", "df_pilot", "default_decode_records"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
print("GPU memory reserved:", f"{torch.cuda.memory_reserved()/1e9:.2f} GB")

GPU memory reserved: 6.92 GB


### LeBron James Triple Retrieval

In [41]:
import json
from collections import defaultdict
from utils.graph_utils import build_graph

target_qid = "WebQTest-268"
rec = None
with open(PLANNING_CKPT) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] == target_qid:
            rec = r
            break

G = build_graph(rec["graph"])
topic = rec["q_entity"][0]  # "LeBron James"

# Group every real 1-hop edge (in either direction) touching the topic entity,
# by relation type -- this is your pool of "Batman / Alfred / Chris Nolan"-style
# context facts for a richer illustrative figure.
outgoing = defaultdict(list)
for nbr in G.neighbors(topic):
    rel = G[topic][nbr]["relation"]
    outgoing[rel].append(nbr)

incoming = defaultdict(list)
for src in G.nodes():
    if src == topic:
        continue
    if G.has_edge(src, topic):
        rel = G[src][topic]["relation"]
        incoming[rel].append(src)

print(f"=== Outgoing relations from '{topic}' ({len(outgoing)} distinct relations) ===")
for rel, targets in sorted(outgoing.items(), key=lambda kv: -len(kv[1])):
    sample = targets[:5]
    print(f"  [{len(targets):>3}x] {rel}  ->  {sample}{' ...' if len(targets) > 5 else ''}")

print(f"\n=== Incoming relations to '{topic}' ({len(incoming)} distinct relations) ===")
for rel, sources in sorted(incoming.items(), key=lambda kv: -len(kv[1])):
    sample = sources[:5]
    print(f"  [{len(sources):>3}x] {rel}  <-  {sample}{' ...' if len(sources) > 5 else ''}")


target_qid = "WebQTest-268"
rec = None
with open(PLANNING_CKPT) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] == target_qid:
            rec = r
            break

G = build_graph(rec["graph"])
topic = rec["q_entity"][0]  # "LeBron James"

# Group every real 1-hop edge (in either direction) touching the topic entity,
# by relation type -- this is your pool of "Batman / Alfred / Chris Nolan"-style
# context facts for a richer illustrative figure.
outgoing = defaultdict(list)
for nbr in G.neighbors(topic):
    rel = G[topic][nbr]["relation"]
    outgoing[rel].append(nbr)

incoming = defaultdict(list)
for src in G.nodes():
    if src == topic:
        continue
    if G.has_edge(src, topic):
        rel = G[src][topic]["relation"]
        incoming[rel].append(src)

print(f"=== Outgoing relations from '{topic}' ({len(outgoing)} distinct relations) ===")
for rel, targets in sorted(outgoing.items(), key=lambda kv: -len(kv[1])):
    sample = targets[:5]
    print(f"  [{len(targets):>3}x] {rel}  ->  {sample}{' ...' if len(targets) > 5 else ''}")

print(f"\n=== Incoming relations to '{topic}' ({len(incoming)} distinct relations) ===")
for rel, sources in sorted(incoming.items(), key=lambda kv: -len(kv[1])):
    sample = sources[:5]
    print(f"  [{len(sources):>3}x] {rel}  <-  {sample}{' ...' if len(sources) > 5 else ''}")

=== Outgoing relations from 'LeBron James' (64 distinct relations) ===
  [ 26x] award.award_winner.awards_won  ->  ['m.0_qrm17', 'm.0yg0zky', 'm.0z66wqh', 'm.0_qrd1p', 'm.0_qrgh9'] ...
  [ 23x] award.award_nominee.award_nominations  ->  ['m.0sgkpd0', 'm.0z1p_l5', 'm.0z9ljvh', 'm.0z5blfv', 'm.0y_yc47'] ...
  [ 20x] award.award_nomination.award_nominee  ->  ['m.0z1n6dn', 'm.0_spmjw', 'm.010w6vqp', 'm.0z43z7_', 'm.0z3v44c'] ...
  [ 16x] award.award_honor.award_winner  ->  ['m.0_qvzz1', 'm.0_qr864', 'm.0x0zlfg', 'm.0_qrlpv', 'm.0x0z10c'] ...
  [ 13x] film.personal_film_appearance.person  ->  ['m.0v4mj0y', 'm.0v4ndsn', 'm.0v4mlsf', 'm.0v462vn', 'm.0v4mk_2'] ...
  [  9x] tv.tv_guest_role.actor  ->  ['m.0y7ls9m', 'm.0y7htk3', 'm.0kb00b3', 'm.0y7j8ry', 'm.0y7klbl'] ...
  [  9x] freebase.valuenotation.is_reviewed  ->  ['Parents', 'Date of birth', 'Children', 'Place of birth', 'Weight'] ...
  [  8x] common.topic.webpage  ->  ['m.0kg5v20', 'm.0bnsx2n', 'm.09wlf17', 'm.09ymk3w', 'm.09x22v7'] ...
 

In [42]:
import json
from collections import defaultdict
from utils.graph_utils import build_graph

target_qid = "WebQTest-268"
rec = None
with open(PLANNING_CKPT) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] == target_qid:
            rec = r
            break

G = build_graph(rec["graph"])
topic = rec["q_entity"][0]  # "LeBron James"

# Group every real 1-hop edge (in either direction) touching the topic entity,
# by relation type -- this is your pool of "Batman / Alfred / Chris Nolan"-style
# context facts for a richer illustrative figure.
outgoing = defaultdict(list)
for nbr in G.neighbors(topic):
    rel = G[topic][nbr]["relation"]
    outgoing[rel].append(nbr)

incoming = defaultdict(list)
for src in G.nodes():
    if src == topic:
        continue
    if G.has_edge(src, topic):
        rel = G[src][topic]["relation"]
        incoming[rel].append(src)

print(f"=== Outgoing relations from '{topic}' ({len(outgoing)} distinct relations) ===")
for rel, targets in sorted(outgoing.items(), key=lambda kv: -len(kv[1])):
    sample = targets[:5]
    print(f"  [{len(targets):>3}x] {rel}  ->  {sample}{' ...' if len(targets) > 5 else ''}")

print(f"\n=== Incoming relations to '{topic}' ({len(incoming)} distinct relations) ===")
for rel, sources in sorted(incoming.items(), key=lambda kv: -len(kv[1])):
    sample = sources[:5]
    print(f"  [{len(sources):>3}x] {rel}  <-  {sample}{' ...' if len(sources) > 5 else ''}")

=== Outgoing relations from 'LeBron James' (64 distinct relations) ===
  [ 26x] award.award_winner.awards_won  ->  ['m.0_qrm17', 'm.0yg0zky', 'm.0z66wqh', 'm.0_qrd1p', 'm.0_qrgh9'] ...
  [ 23x] award.award_nominee.award_nominations  ->  ['m.0sgkpd0', 'm.0z1p_l5', 'm.0z9ljvh', 'm.0z5blfv', 'm.0y_yc47'] ...
  [ 20x] award.award_nomination.award_nominee  ->  ['m.0z1n6dn', 'm.0_spmjw', 'm.010w6vqp', 'm.0z43z7_', 'm.0z3v44c'] ...
  [ 16x] award.award_honor.award_winner  ->  ['m.0_qvzz1', 'm.0_qr864', 'm.0x0zlfg', 'm.0_qrlpv', 'm.0x0z10c'] ...
  [ 13x] film.personal_film_appearance.person  ->  ['m.0v4mj0y', 'm.0v4ndsn', 'm.0v4mlsf', 'm.0v462vn', 'm.0v4mk_2'] ...
  [  9x] tv.tv_guest_role.actor  ->  ['m.0y7ls9m', 'm.0y7htk3', 'm.0kb00b3', 'm.0y7j8ry', 'm.0y7klbl'] ...
  [  9x] freebase.valuenotation.is_reviewed  ->  ['Parents', 'Date of birth', 'Children', 'Place of birth', 'Weight'] ...
  [  8x] common.topic.webpage  ->  ['m.0kg5v20', 'm.0bnsx2n', 'm.09wlf17', 'm.09ymk3w', 'm.09x22v7'] ...
 